#### Import Libraries

In [20]:
from dotenv import load_dotenv
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

#### Load Environment & LLM

In [21]:
load_dotenv()
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)

#### Load PDF

In [22]:
# Update this path to your PDF file
loader = PyPDFLoader(r"F:\GenAI\datasets\1.Introduction to sequential data and RNNs.pdf")
docs = loader.load()
print(f"Loaded {len(docs)} pages")

Loaded 2 pages


#### Chunk Documents

In [23]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(docs)
print(f"Total chunks: {len(chunks)}")

Total chunks: 8


#### Embeddings + FAISS Vector Store

In [24]:
emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vs = FAISS.from_documents(chunks, emb)
retriever = vs.as_retriever(search_kwargs={"k": 3})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

#### Building Conversational RAG

In [25]:
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# Prompt includes chat history for memory
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the context below to answer the user's question.\n\nContext:\n{context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

# KEY FIX: extract question string for retriever, keep full dict for prompt
rag_chain = (
    RunnablePassthrough.assign(
        context=RunnableLambda(lambda x: x["question"]) | retriever | format_docs
    )
    | prompt
    | llm
    | StrOutputParser()
)

# In-memory store for multiple sessions
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

conversational_rag = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)

c:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\IPython\core\interactiveshell.py:3699: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


#### Test the Chain — Turn 1

In [27]:
response1 = conversational_rag.invoke(
    {"question": "What is RNN?"},
    config={"configurable": {"session_id": "user1"}}
)
print("Turn 1:", response1)

Turn 1: A Recurrent Neural Network (RNN) is a special type of neural network designed to process sequential data, where the order of information is important. Unlike traditional feedforward networks, RNNs have connections that loop back, allowing information from previous steps to influence the current output. This looping structure helps the network retain a form of memory, enabling it to learn.


#### Test the Chain — Turn 2 (Memory Test)

In [29]:
response2 = conversational_rag.invoke(
    {"question": "What is Sequential Data?"},
    config={"configurable": {"session_id": "user1"}}
)
print("Turn 2:", response2)

Turn 2: Sequential data refers to data in which the order of elements matters, and each data point is dependent on the ones that came before it. Unlike traditional datasets where observations are independent, sequential data captures patterns, context, and dependencies across time or position.

Examples of sequential data include:
*   **Text:** (e.g., next word prediction, sentiment analysis)
*   **Time Series:** (e.g., stock prices, weather forecasting)
*   **Audio:** (e.g., speech recognition, music generation)
*   **Video:** (e.g., action recognition, captioning)


#### Test the Chain — Turn 3 (Refers Back to Previous Answer)

In [30]:
response3 = conversational_rag.invoke(
    {"question": "How it is used in RNN?"},
    config={"configurable": {"session_id": "user1"}}
)
print("Turn 3:", response3)

Turn 3: RNNs are specifically designed to process sequential data by leveraging their unique architecture. Here's how they use it:

1.  **Processing Order:** RNNs are built to handle data where the order of information is crucial. They don't treat each piece of data in a sequence as independent but rather consider its relationship to previous elements.
2.  **Memory through Looping Connections:** Unlike traditional neural networks, RNNs have connections that loop back. This allows information from a previous step in the sequence to be fed back into the network as it processes the current step.
3.  **Influencing Current Output:** This "looping back" mechanism means that the network's understanding and output for the current element in the sequence are influenced by the information it processed from the preceding elements.
4.  **Retaining Context/Memory:** This structure enables the RNN to retain a form of "memory" or context about the sequence it has seen so far. This memory is crucial f